# 反向探针：给出位置，能读出物体类别吗

**问题：** 之前的探针给物体名、读位置，结果探针按名字找不到物体。这里反过来：告诉探针物体在哪里，让它预测类别名。这样可以区分两种情况：token 里本来就没有类别信息，或者有但按名字检索不到。

**设置**（数据、划分、训练设置与位置探针相同，代码在 `train_reverse.py`）：

- **目标：** 物体的类别，在训练集出现过的类别中做分类（交叉熵）。物体与点提示实验相同：场景中只出现一次的类别，且框中心至少落在一帧画面内。验证集里类别没在训练集出现过的物体不计入。
- **注意力读出（三种 token 都适用）：** 与位置探针相同的 2 层 cross-attention。查询只有物体中心像素的 Plücker 射线（Fourier 编码），不含任何图像特征，所以类别只能来自被读取的 token。
- **直接索引读出（只适用于 DINOv2 和 VGGT）：** 直接取物体中心所在的那一个 patch token，接 MLP 分类。不需要检索，衡量的是这个 token 本身含有多少类别信息。3DRAE 的 token 没有对应的像素位置，做不了这一项。
- **下限：** 训练时把 token 换成其他场景的（只靠射线能猜到的先验），以及永远猜最常见类别。

**怎么读：** top-1 / top-5 是按物体平均的准确率，方括号内为按场景 bootstrap 的 95% 置信区间；macro top-1 是先按类别求准确率再平均，不受常见类别主导。每组一个随机种子。

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "outputs" / "geo_probe").is_dir())
RUNS = ROOT / "outputs" / "geo_probe"
pd.set_option("display.width", 200)

NAMES = {"3DRAE, attention": "reverse_3drae_real", "DINOv2 patch, attention": "reverse_dino_real",
         "VGGT, attention": "reverse_vggt_real", "DINOv2 patch, index": "reverse_index_dino",
         "VGGT, index": "reverse_index_vggt", "shuffled 3DRAE (floor)": "reverse_3drae_shuffled"}
COLORS = {"3DRAE, attention": "#2a78d6", "DINOv2 patch, attention": "#8a8f98", "VGGT, attention": "#eb6834",
          "DINOv2 patch, index": "#4a4f57", "VGGT, index": "#b23c0c", "shuffled 3DRAE (floor)": "#c8ccd2"}
FREQ = [(1, 9, "1-9"), (10, 49, "10-49"), (50, 199, "50-199"), (200, 10**9, "200+")]


def final(run):
    path = RUNS / run / "final_metrics.json"
    return json.loads(path.read_text()) if path.exists() else None


def objects(run):
    path = RUNS / run / "objects.jsonl"
    if not path.exists():
        return None
    frame = pd.DataFrame([json.loads(line) for line in path.open()])
    frame = frame[frame.train_freq > 0].copy()
    frame["top1"] = [r.top5[0] == r.category for r in frame.itertuples()]
    frame["in_top5"] = [r.category in r.top5 for r in frame.itertuples()]
    return frame


RESULTS = {name: final(run) for name, run in NAMES.items() if final(run)}
rows = {}
for name, result in RESULTS.items():
    e = result["eval_real"]
    row = {key: f"{e[key]['value'] * 100:.1f} [{e[key]['ci95'][0] * 100:.1f}, {e[key]['ci95'][1] * 100:.1f}]"
           for key in ("top1", "top5")}
    row["macro top1"] = f"{e['macro_top1'] * 100:.1f}"
    s = result.get("eval_shuffled")
    row["top1, tokens swapped at eval"] = f"{s['top1']['value'] * 100:.1f}" if s else "-"
    row["n objects"] = e["n"]
    rows[name] = row
any_result = next(iter(RESULTS.values()))
print(f"{any_result['classes']} train categories; always guessing '{any_result['majority_class']}' gives "
      f"top-1 {any_result['majority_top1'] * 100:.1f}%; {any_result['eval_real']['n_unseen_category']} val objects "
      f"of unseen categories left out")
pd.DataFrame(rows).T

## 1. 总体准确率

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
names = list(RESULTS)
for ax, key in zip(axes, ("top1", "top5", "macro_top1")):
    e = [RESULTS[n]["eval_real"] for n in names]
    if key == "macro_top1":
        y, err = np.array([m[key] for m in e]) * 100, None
    else:
        y = np.array([m[key]["value"] for m in e]) * 100
        err = np.array([[m[key]["value"] - m[key]["ci95"][0], m[key]["ci95"][1] - m[key]["value"]] for m in e]).T * 100
    ax.bar(range(len(y)), y, yerr=err, color=[COLORS[n] for n in names], capsize=3)
    for i, v in enumerate(y):
        ax.text(i, v + (err[1][i] if err is not None else 0), f"{v:.1f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(range(len(y)), [n.replace(", ", "\n").replace(" (", "\n(") for n in names], fontsize=7)
    ax.set_title({"top1": "top-1 acc (%)", "top5": "top-5 acc (%)", "macro_top1": "macro top-1 acc (%)"}[key], fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## 2. 按类别在训练集中的出现次数

In [ ]:
OBJECTS = {name: objects(run) for name, run in NAMES.items() if objects(run) is not None}
table = {}
for name, frame in OBJECTS.items():
    table[name] = {f"{label} (n={((frame.train_freq >= lo) & (frame.train_freq <= hi)).sum()})":
                   frame.top1[(frame.train_freq >= lo) & (frame.train_freq <= hi)].mean() * 100 for lo, hi, label in FREQ}
display(pd.DataFrame(table).T.round(1))

fig, ax = plt.subplots(figsize=(6.5, 3.6))
for name, frame in OBJECTS.items():
    y = [frame.top1[(frame.train_freq >= lo) & (frame.train_freq <= hi)].mean() * 100 for lo, hi, _ in FREQ]
    ax.plot(range(len(FREQ)), y, marker="o", color=COLORS[name], label=name,
            linestyle="--" if "shuffled" in name else (":" if "index" in name else "-"))
ax.set_xticks(range(len(FREQ)), [b[2] for b in FREQ])
ax.set_xlabel("category occurrences in train")
ax.set_ylabel("top-1 acc (%)")
ax.set_ylim(0, 100)
ax.legend(fontsize=8, frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 3. 验证集中最常见的 20 个类别的 top-1 准确率（%）

In [ ]:
first = next(iter(OBJECTS.values()))
common = first.category.value_counts().head(20)
per_category = pd.DataFrame({name: frame.groupby("category").top1.mean().reindex(common.index) * 100
                             for name, frame in OBJECTS.items()}).round(0)
per_category.insert(0, "n", common.values)
per_category

## 4. 训练曲线

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for name, run in NAMES.items():
    path = RUNS / run / "metrics.jsonl"
    if not path.exists():
        continue
    log = pd.DataFrame([json.loads(line) for line in path.open()]).drop_duplicates("epoch", keep="last")
    style = "--" if "shuffled" in name else (":" if "index" in name else "-")
    axes[0].plot(log.epoch, log.loss, color=COLORS[name], linestyle=style, label=f"{name} (final {log.loss.iloc[-1]:.2f})")
    axes[1].plot(log.epoch, log.train_top1 * 100, color=COLORS[name], linestyle=style,
                 label=f"{name} (final {log.train_top1.iloc[-1] * 100:.0f}%)")
axes[0].set_yscale("log")
for ax, label in zip(axes, ("train loss", "train top-1 acc (%)")):
    ax.set_xlabel("epoch")
    ax.set_ylabel(label)
    ax.legend(fontsize=7, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## 5. 结论

（实验完成后填写）